# Huấn luyện thêm YOLO26n trên Colab (Nhóm 14)
**Runtime → Change runtime type → T4 GPU**

Mục tiêu: tăng độ chính xác so với mô hình hiện tại (Test: mAP50 0.922, mAP50-95 0.820), nhất là 2 lớp yếu **TH true MILK** và **Handy Hảo Hảo**.

Hướng tiếp cận: không học lại từ đầu mà **xuất phát từ `models/yolo26n/weights/best.pt`**, dùng lr nhỏ (0.0005, AdamW) + tăng cường dữ liệu mạnh hơn (xoay ±10°, mixup). Chạy 2 lượt:
- `ft640`: giữ ảnh 640 như ứng dụng đang dùng.
- `ft832`: ảnh 832, khung bao chính xác hơn nhưng chậm hơn (nếu chọn lượt này phải đổi `app.imgsz` thành 832).

Mọi tham số ở mục `detection.finetune` của `configs/config.yaml`. Mô hình cũ **không bị ghi đè**, mỗi lượt lưu ở `models/yolo26n_<lượt>/`.

## 1. Kết nối Drive và lấy mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Thư mục trên Drive chứa data/yolo và nơi lưu kết quả
DRIVE_YOLO = '/content/drive/MyDrive/yolo'
DRIVE_OUT  = '/content/drive/MyDrive/NHOM_14_ket_qua_yolo/huan_luyen_them'
REPO_URL   = 'https://github.com/hoangtrongkhoi050515/Product-recognition-in-supermarkets.git'
BRANCH     = 'main'   # nhánh đã có src/detection/train.py bản mới (--finetune)

In [ ]:
%cd /content
!rm -rf repo
!git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd /content/repo
!pip install -q "ultralytics>=8.4" pyyaml pandas
!ls -la models/yolo26n/weights/best.pt   # mô hình cũ, điểm xuất phát

## 2. Chép dữ liệu về ổ Colab và sinh lại `data.yaml`
Số ảnh phải đúng **682 / 151 / 151** (train / val / test) như lần huấn luyện đầu, nếu khác thì so sánh không công bằng.

In [ ]:
!mkdir -p data
!rm -rf data/yolo && cp -r {DRIVE_YOLO} data/yolo
!rm -f data/yolo/data.yaml
!python -m src.data.split_dataset --yaml-only
!ls data/yolo/images/train | wc -l ; ls data/yolo/images/val | wc -l ; ls data/yolo/images/test | wc -l

## 3. Đánh giá lại mô hình cũ trên cùng dữ liệu (mốc so sánh)
Kết quả tập Test phải xấp xỉ số trong báo cáo (mAP50 0.922, mAP50-95 0.820). Nếu lệch nhiều thì dữ liệu trên Drive đã khác lần trước, cần kiểm tra trước khi huấn luyện.

In [ ]:
for s in ['val', 'test']:
    !python -m src.detection.evaluate --run yolo26n --split {s}

## 4. Chạy thử 1 epoch trên 5% dữ liệu (kiểm tra lỗi, ~1 phút)

In [ ]:
!python -m src.detection.train --finetune ft640 --epochs 1 --fraction 0.05

## 5. Huấn luyện thêm (tối đa 80 epoch, dừng sớm sau 30 epoch không cải thiện)
Sau mỗi lượt chép ngay kết quả về Drive để lỡ Colab ngắt kết nối thì không mất.

In [ ]:
!python -m src.detection.train --finetune ft640
!mkdir -p {DRIVE_OUT}/models && cp -r models/yolo26n_ft640 {DRIVE_OUT}/models/

In [ ]:
!python -m src.detection.train --finetune ft832
!mkdir -p {DRIVE_OUT}/models && cp -r models/yolo26n_ft832 {DRIVE_OUT}/models/

## 6. Đánh giá các lượt mới trên tập Validation và Test
Mỗi lượt đánh giá ở đúng kích thước ảnh đã huấn luyện.

In [ ]:
RUNS = {'yolo26n_ft640': 640, 'yolo26n_ft832': 832}
for run, size in RUNS.items():
    for s in ['val', 'test']:
        !python -m src.detection.evaluate --run {run} --split {s} --imgsz {size}

## 7. So sánh với mô hình cũ
**Chọn mô hình theo tập Validation** (mAP50-95), tập Test chỉ dùng để báo cáo. Cột Δ là chênh lệch so với mô hình cũ.

In [ ]:
!python -m src.detection.compare --runs yolo26n yolo26n_ft640 yolo26n_ft832

In [ ]:
# Ma trận nhầm lẫn tập Test: xem Hảo Hảo có còn bị nhầm với Tomyum / bị bỏ sót không
from IPython.display import Image, display
for run in ['yolo26n', 'yolo26n_ft640', 'yolo26n_ft832']:
    print(run)
    display(Image(f'Logs/detection/{run}_test/confusion_matrix_normalized.png', width=700))

## 8. Chép toàn bộ kết quả về Drive

In [ ]:
!mkdir -p {DRIVE_OUT}
!cp -r models/yolo26n_ft640 models/yolo26n_ft832 {DRIVE_OUT}/models/ 2>/dev/null
!cp -r Logs/detection {DRIVE_OUT}/
!python -m src.detection.compare --runs yolo26n yolo26n_ft640 yolo26n_ft832 > {DRIVE_OUT}/so_sanh.txt
print('Đã chép về', DRIVE_OUT)